# 8. Hafta — Sınıflandırma Algoritmaları ve Veri Ön İşleme

**BTP271 Uygulamalı Yapay Zeka** · Ön-Yüz Yazılım Geliştirme
Öğr. Gör. Çisem YAŞAR · cyasar@gelisim.edu.tr

### Bu notebook'taki alıştırma türleri

| İşaret | Tür | Ne yapacaksın? |
|---|---|---|
| ✍️ | **Kendin Yaz** | Kodu sıfırdan yaz. |
| 🧪 | **Test Hücresi** | Kodunu yazdıktan sonra çalıştır. `✅ Tüm testler geçti!` görürsen sonucun doğru. |

> 💡 Bu haftanın sonunda **proje 2. aşamasının** tüm adımlarını görmüş olacaksın: ayırma, ölçeklendirme, algoritma seçimi, parametreler, metrikler ve modeli kaydetme.

---
## 0. Veri ve ayrım (7. haftadakiyle aynı)
Sadece çalıştır. Ayrım parametreleri 7. haftayla **aynı** olduğu için sonuçları geçen haftayla karşılaştırabileceğiz.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

veri = load_breast_cancer(as_frame=True)
df = veri.frame
df["kotu_huylu"] = 1 - df["target"]          # 1 = kötü huylu
X = df[veri.feature_names]
y = df["kotu_huylu"]

X_egitim, X_test, y_egitim, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def metrikler(gercek, tahmin):
    """Accuracy, Precision, Recall, F1 değerlerini liste olarak döndürür (7. hafta)."""
    return [round(accuracy_score(gercek, tahmin), 3), round(precision_score(gercek, tahmin), 3),
            round(recall_score(gercek, tahmin), 3), round(f1_score(gercek, tahmin), 3)]

print("Eğitim:", X_egitim.shape, "| Test:", X_test.shape)

---
## 1. Ölçeklendirme

7\. haftada fark ettik: `mean area` yüzlerle binler arasında, `mean smoothness` ise 0.05 ile 0.16 arasında.
**Uzaklığa** bakan algoritmalar (ör. KNN) büyük sayılı sütunu çok önemli, küçük sayılıyı önemsiz sanır.
**Ölçeklendirme**, tüm sütunları benzer bir aralığa getirir.

| Yöntem | Ne yapar? | Sonuç |
|---|---|---|
| `StandardScaler` | Ortalamayı çıkarır, standart sapmaya böler | Ortalama 0, standart sapma 1 |
| `MinMaxScaler` | En küçüğü 0, en büyüğü 1 yapar | 0 ile 1 arası (3. haftadaki `/ 255` gibi) |

### ⚠️ Veri sızıntısı: En önemli kural
Ölçekleyici ortalamayı ve standart sapmayı **veriden öğrenir** (`fit`). Bu öğrenme **sadece eğitim verisiyle** yapılmalı.
Test verisine `fit` edersen, model test verisi hakkında önceden bilgi almış olur. Bu, sınav sorularından ipucu sızdırmaya benzer: sonuçlar gerçekte olduğundan iyi görünür.

```python
olcekleyici.fit(X_egitim)                        # ✅ sadece eğitimden öğren
X_egitim_olc = olcekleyici.transform(X_egitim)   # ✅ eğitimi dönüştür
X_test_olc = olcekleyici.transform(X_test)       # ✅ testi AYNI kuralla dönüştür

olcekleyici.fit(X)                               # ❌ tüm veriden öğrenmek = sızıntı
```
> Aynı kural eksik veri doldururken de geçerli: `fillna` için kullandığın medyanı **eğitim verisinden** hesapla.

#### ✍️ Kendin Yaz — StandardScaler
1. `from sklearn.preprocessing import StandardScaler` ile bir `olcekleyici` oluştur ve **sadece** `X_egitim` ile `fit` et.
2. Eğitim ve test verisini dönüştürüp `X_egitim_olc` ve `X_test_olc` değişkenlerine ata.
   *(Sonuç bir NumPy dizisidir; sütun adlarını geri almak için: `pd.DataFrame(X_egitim_olc, columns=X.columns)`)*
3. `"mean area"` ve `"mean smoothness"` sütunlarının ortalamasını ve standart sapmasını **ölçeklendirmeden önce ve sonra** yazdır.

Yorum satırına cevapla: Ölçeklendirmeden sonra iki sütunun değerleri neden artık karşılaştırılabilir?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi — değiştirme, sadece çalıştır
assert X_egitim_olc.shape == (455, 30) and X_test_olc.shape == (114, 30)
assert abs(X_egitim_olc[:, 3].mean()) < 1e-9          # "mean area" eğitimde ortalama 0
assert abs(olcekleyici.mean_[3] - X_egitim["mean area"].mean()) < 1e-6   # sadece eğitimden öğrenildi
print("✅ Tüm testler geçti!")

---
## 2. KNN (K-En Yakın Komşu)

Yeni bir hastaya bakarken, eğitim verisinde **ona en çok benzeyen k hastayı** bulur ve onların çoğunluğu neyse onu tahmin eder.
- "Benzerlik" = sütunlar arasındaki **uzaklık**. Bu yüzden KNN ölçeklendirmeye çok duyarlıdır.
- `n_neighbors` (k): kaç komşuya bakılacağı. Bu, bir **hiperparametredir**: modelin veriden öğrenmediği, bizim seçtiğimiz bir ayar.

#### ✍️ Kendin Yaz — Ölçeklendirme KNN'yi nasıl etkiliyor?
1. `KNeighborsClassifier(n_neighbors=5)` modelini **ölçeklenmemiş** veriyle (`X_egitim`) eğit, test metriklerini `ham` değişkenine ata. *(Hazır `metrikler` fonksiyonunu kullan.)*
2. Aynı modeli **ölçeklenmiş** veriyle (`X_egitim_olc`) eğit, test metriklerini `olcekli` değişkenine ata.
3. İkisini karşılaştıran bir tablo yazdır (sütunlar: Accuracy, Precision, Recall, F1).

Yorum satırına cevapla: Hangi metrik en çok değişti? Bir kanser taraması için bu ne anlama geliyor?

In [ ]:
# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
assert len(ham) == 4 and len(olcekli) == 4
assert olcekli[0] > ham[0]          # doğruluk arttı
assert olcekli[2] > ham[2]          # recall arttı
print("✅ Tüm testler geçti!")

#### ✍️ Kendin Yaz — En iyi k değeri hangisi?
1. `[1, 3, 5, 7, 9, 11, 15, 21]` değerlerinin her biri için ölçeklenmiş veriyle bir KNN eğit, **test F1** değerini `f1_listesi`'ne ekle.
2. `plt.plot` ile k–F1 grafiğini çiz (`marker="o"`).
3. F1'i en yüksek olan **ilk** k değerini `en_iyi_k` değişkenine ata ve yazdır.

Yorum satırına cevapla: k = 1 neden iyi sonuç vermiyor olabilir? (İpucu: tek bir komşuya bakmak ne kadar güvenilir?)

In [ ]:
k_degerleri = [1, 3, 5, 7, 9, 11, 15, 21]

# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
assert len(f1_listesi) == len(k_degerleri)
assert en_iyi_k in k_degerleri and f1_listesi[k_degerleri.index(en_iyi_k)] == max(f1_listesi)
print("✅ Tüm testler geçti!")

---
## 3. Karar Ağacı

Veriye **evet / hayır soruları** sorarak karar verir: *"worst perimeter ≤ 112.8 mi? Evet → worst concave points ≤ 0.16 mı? …"*
- Ölçeklendirmeye **ihtiyaç duymaz** (her soru tek bir sütuna bakar).
- En büyük avantajı: **okunabilir.** Kuralları bir insan takip edebilir. 5. haftada elle yazdığımız eşik kurallarının otomatik halidir.
- `max_depth`: ağacın en fazla kaç soru derinliğe inebileceği.

Aşağıda 2 derinlikli küçük bir ağacı çizdiriyoruz. Sadece çalıştır ve bir hastanın yolunu kökten yaprağa takip et.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

kucuk_agac = DecisionTreeClassifier(max_depth=2, random_state=42)
kucuk_agac.fit(X_egitim, y_egitim)

plt.figure(figsize=(12, 6))
plot_tree(kucuk_agac, feature_names=list(X.columns), class_names=["iyi huylu", "kötü huylu"],
          filled=True, rounded=True, fontsize=10)
plt.title("2 derinlikli karar ağacı")
plt.show()

#### ✍️ Kendin Yaz — Ağaç derinleştikçe ne oluyor?
1. `max_depth` için `[1, 2, 3, 4, 5, 6, None]` değerlerini dene (`None` = sınırsız). Her biri için `DecisionTreeClassifier(max_depth=d, random_state=42)` eğit.
2. Her derinlik için **eğitim** ve **test** doğruluğunu `egitim_dog` ve `test_dog` listelerine ekle.
3. İki çizgiyi aynı grafikte göster. x ekseni etiketleri için: `etiketler = [str(d) for d in derinlikler]`

Yorum satırına cevapla: Sınırsız ağacın eğitim doğruluğu kaç? Test doğruluğuyla arasındaki fark ne anlatıyor?

In [ ]:
derinlikler = [1, 2, 3, 4, 5, 6, None]

# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
assert len(egitim_dog) == len(test_dog) == 7
assert egitim_dog[-1] == 1.0                 # sınırsız ağaç eğitimi ezberliyor
assert egitim_dog[-1] > test_dog[-1]
print("✅ Tüm testler geçti!")

---
## 4. Naive Bayes ve Random Forest

**Naive Bayes:** Her ölçümün sınıfa göre dağılımına bakarak **olasılık** hesaplar. Öznitelikleri birbirinden bağımsız varsayar ("naive" = saf). Çok hızlıdır; özellikle metin sınıflandırmada (13. hafta) sık kullanılır.

**Random Forest (rastgele orman):** Her biri verinin farklı bir parçasıyla eğitilmiş **yüzlerce karar ağacı** kurar ve ağaçların **oylamasıyla** karar verir. Tek bir ağacın ezberleme sorununu büyük ölçüde çözer.

Aşağıdaki hücreyi sadece çalıştır.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

nb = GaussianNB()
nb.fit(X_egitim, y_egitim)

rf = RandomForestClassifier(n_estimators=100, random_state=42)   # 100 ağaç
rf.fit(X_egitim, y_egitim)

print(pd.DataFrame([metrikler(y_test, nb.predict(X_test)), metrikler(y_test, rf.predict(X_test))],
                   index=["Naive Bayes", "Random Forest"], columns=["Accuracy", "Precision", "Recall", "F1"]))

### Hangi ölçümler daha önemli?
Random Forest her sütunun kararlara ne kadar katkı verdiğini hesaplar. Proje raporunda modelini yorumlarken işine yarar.

In [ ]:
onem = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(8)

plt.barh(onem.index, onem.values)
plt.title("Random Forest: en önemli 8 öznitelik")
plt.xlabel("Önem")
plt.show()

---
## 5. Modeli kaydetmek

Eğittiğin modeli her seferinde yeniden eğitmek istemezsin. Web uygulaması da modeli bir **dosyadan** yükler.
`joblib` ile model bir dosyaya kaydedilir ve istendiğinde geri yüklenir.

> ⚠️ Model ölçeklenmiş veriyle eğitildiyse **ölçekleyiciyi de kaydetmelisin.** Yeni bir hasta geldiğinde önce aynı ölçekleyiciden geçmeli.

In [ ]:
import joblib

joblib.dump(knn_olc, "knn_model.joblib")          # modeli kaydet
joblib.dump(olcekleyici, "olcekleyici.joblib")    # ölçekleyiciyi de kaydet

# … başka bir gün, başka bir programda:
yuklu_model = joblib.load("knn_model.joblib")
yuklu_olcekleyici = joblib.load("olcekleyici.joblib")

yeni_hasta = X_test.iloc[[0]]                     # tek satırlık tablo
tahmin = yuklu_model.predict(yuklu_olcekleyici.transform(yeni_hasta))
print("Tahmin:", "kötü huylu" if tahmin[0] == 1 else "iyi huylu", "| Gerçek:", y_test.iloc[0])

### 🖼️ Projende görsel veri kullanıyorsan
Scikit-learn modelleri **tablo** bekler: her satır bir örnek, her sütun bir sayı. Bir görüntü ise 2 boyutlu bir sayı dizisidir (3. hafta).
Görüntüyü klasik ML modeline vermek için **düzleştirilir**: 28×28 piksellik bir görüntü, 784 sütunlu tek bir satır olur.

```python
# goruntuler.shape → (1000, 28, 28)   : 1000 görüntü, her biri 28×28
X_duz = goruntuler.reshape(len(goruntuler), -1)
# X_duz.shape → (1000, 784)           : artık bir tablo; ölçekle (/ 255) ve modele ver
```
Derin öğrenmede (12. hafta, CNN) bu düzleştirmeye gerek kalmayacak: CNN görüntüyü kendi 2 boyutlu yapısıyla işler.

---
## 🏁 Haftanın Mini Projesi — Model Yarışması (Proje 2. Aşama Provası)

Beş algoritmayı **aynı** eğitim / test ayrımıyla karşılaştır, en iyisini seç ve kaydet.

**Görevler:**
1. Aşağıdaki beş modeli eğit ve her birinin **test** metriklerini bir `tablo` DataFrame'inde topla (satırlar model adları, sütunlar Accuracy, Precision, Recall, F1):
   - `"Lojistik Regresyon"` → `LogisticRegression(max_iter=5000)`, **ölçeklenmiş** veriyle
   - `"KNN"` → `KNeighborsClassifier(n_neighbors=en_iyi_k)`, **ölçeklenmiş** veriyle
   - `"Karar Ağacı"` → `DecisionTreeClassifier(random_state=42)`
   - `"Naive Bayes"` → `GaussianNB()`
   - `"Random Forest"` → `RandomForestClassifier(n_estimators=100, random_state=42)`
   *(İpucu: modelleri bir sözlükte tut, döngüyle eğit.)*
2. Tabloyu **Recall'a göre** büyükten küçüğe sırala; eşitlikte F1'e bak: `tablo.sort_values(["Recall", "F1"], ascending=False)`. İlk satırdaki modelin adını `en_iyi` değişkenine ata.
3. En iyi modeli `en_iyi_model.joblib` olarak kaydet; geri yükle ve test verisindeki **ilk 3 hasta** için tahmin ve kötü huylu olasılığını JSON olarak yazdır.
   *(Model ölçeklenmiş veriyle eğitildiyse yükledikten sonra `olcekleyici`'den geçirmeyi unutma.)*

Yorum satırına cevapla: Neden Recall'a göre sıraladık? Seçtiğin modelin eğitimde kullanılan temel parametreleri neler?

In [ ]:
import json
from sklearn.linear_model import LogisticRegression

# kodunu yaz


# Cevabım:

In [ ]:
# 🧪 Test hücresi
import os
assert tablo.shape == (5, 4)
assert set(tablo.index) == {"Lojistik Regresyon", "KNN", "Karar Ağacı", "Naive Bayes", "Random Forest"}
assert tablo.loc[en_iyi, "Recall"] == tablo["Recall"].max()
assert os.path.exists("en_iyi_model.joblib")
assert len(sonuc) == 3
print("✅ Tüm testler geçti! Tebrikler 🎉")

---
### 📌 Haftaya Hazırlık
- Bitiremediğin alıştırmaları ve mini projeyi tamamla.
- **Proje 2. aşama:** Kendi veri setinde ölçeklendirmeyi (sadece eğitime `fit`!) uygula, en az iki algoritmayı karşılaştır, seçtiğin modelin parametrelerini ve metriklerini raporla, modeli `joblib` ile kaydet.
- Gelecek hafta: **Yapay sinir ağlarına giriş.** Bir nöron nasıl "öğrenir"? Perseptronu bu veri setiyle kendimiz eğiteceğiz.